# Standard UNet for R-Peak Detection on MIT-BIH Arrhythmia Database

This notebook implements a standard UNet architecture (without dense skip connections) specifically designed to detect R-peaks in long-term ECG signals from the MIT-BIH Arrhythmia Database.

**Key Features:**
1. **Inter-Patient Split**: Uses the research gold-standard DS1 (Train) and DS2 (Test) split.
2. **Mask Generation**: Widens single-point R-peak annotations into a $\pm 10$ sample region ($55\text{ms}$) to create a binary segmentation mask.
3. **Chunking**: Chops continuous 30-minute signals into non-overlapping 2048-sample segments.
4. **UNet Architecture**: Standard UNet with encoder, bottleneck, and symmetric decoder.
5. **Evaluation**: Outputs standard metrics (Sensitivity, Positive Predictivity, Error Rate) directly comparable to the UNet++ and Wavelet methods.



In [1]:
import wfdb
import numpy as np
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd
import time
import scipy.signal as signal

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

# GPU Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


## 1. Data Processing and Mask Generation


In [2]:
# Standard MIT-BIH Inter-Patient Split (DS1 for train, DS2 for test)
DS1_TRAIN = ['101', '106', '108', '109', '112', '114', '115', '116', '118', '119', '122', '124', 
             '201', '203', '205', '207', '208', '209', '215', '220', '223', '230']
DS2_TEST = ['100', '103', '105', '111', '113', '117', '121', '123', '200', '202', '210', '212', 
            '213', '214', '219', '221', '222', '228', '231', '232', '233', '234']

CHUNK_SIZE = 2048
PEAK_RADIUS = 10 # +/- 10 samples around the peak marked as class 1

def get_annotated_peaks(record_path):
    ann = wfdb.rdann(record_path, 'atr')
    beat_types = ['N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r', 'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?']
    beats = [ann.sample[i] for i in range(len(ann.symbol)) if ann.symbol[i] in beat_types]
    return np.array(beats)

def prepare_dataset(records_list):
    X_data = []
    y_data = []
    
    for rec in records_list:
        if not os.path.exists(rec + '.dat'):
            continue
            
        record = wfdb.rdsamp(rec)
        ecg = record[0][:, 0] # channel 0
        peaks = get_annotated_peaks(rec)
        
        mask = np.zeros(len(ecg), dtype=np.int64)
        for p in peaks:
            start = max(0, p - PEAK_RADIUS)
            end = min(len(ecg), p + PEAK_RADIUS + 1)
            mask[start:end] = 1
            
        num_chunks = len(ecg) // CHUNK_SIZE
        for i in range(num_chunks):
            start = i * CHUNK_SIZE
            end = start + CHUNK_SIZE
            
            chunk_ecg = ecg[start:end]
            chunk_mask = mask[start:end]
            
            std = np.std(chunk_ecg)
            if std > 0:
                chunk_ecg = (chunk_ecg - np.mean(chunk_ecg)) / std
            else:
                chunk_ecg = chunk_ecg - np.mean(chunk_ecg)
                
            X_data.append(chunk_ecg)
            y_data.append(chunk_mask)
            
    X_tensor = torch.tensor(np.array(X_data), dtype=torch.float32).unsqueeze(1)
    y_tensor = torch.tensor(np.array(y_data), dtype=torch.long)
    return X_tensor, y_tensor

print("Preparing Training Set (DS1)...")
X_train, y_train = prepare_dataset(DS1_TRAIN)
print("Preparing Test Set (DS2)...")
X_test, y_test = prepare_dataset(DS2_TEST)

print(f"Train Shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test Shape:  X={X_test.shape}, y={y_test.shape}")



Preparing Training Set (DS1)...
Preparing Test Set (DS2)...
Train Shape: X=torch.Size([6974, 1, 2048]), y=torch.Size([6974, 2048])
Test Shape:  X=torch.Size([6974, 1, 2048]), y=torch.Size([6974, 2048])


## 2. Standard UNet Architecture


In [3]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=9, padding=4)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=9, padding=4)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        return x

class ECGUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()

        # Encoder (Downsampling path) - 4 blocks
        self.enc1 = ConvBlock(in_channels, 16)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.enc2 = ConvBlock(16, 32)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.enc3 = ConvBlock(32, 64)
        self.pool3 = nn.MaxPool1d(kernel_size=2)

        self.enc4 = ConvBlock(64, 128)
        self.pool4 = nn.MaxPool1d(kernel_size=2)

        # Bottleneck
        self.bottleneck = ConvBlock(128, 256)

        # Decoder (Upsampling path)
        self.up1 = nn.ConvTranspose1d(256, 128, kernel_size=8, stride=2, padding=3)
        self.dec1 = ConvBlock(256, 128) # 128 (skip) + 128 (up) = 256 input channels

        self.up2 = nn.ConvTranspose1d(128, 64, kernel_size=8, stride=2, padding=3)
        self.dec2 = ConvBlock(128, 64)

        self.up3 = nn.ConvTranspose1d(64, 32, kernel_size=8, stride=2, padding=3)
        self.dec3 = ConvBlock(64, 32)

        self.up4 = nn.ConvTranspose1d(32, 16, kernel_size=8, stride=2, padding=3)
        self.dec4 = ConvBlock(32, 16)

        # Final output layer
        self.out_conv = nn.Conv1d(16, num_classes, kernel_size=1)

    def pad_to_match(self, x, skip_connection):
        diff = skip_connection.size(-1) - x.size(-1)
        if diff > 0:
            x = F.pad(x, (0, diff))
        return x

    def forward(self, x):
        # Downsample
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        e4 = self.enc4(p3)
        p4 = self.pool4(e4)

        # Bottleneck
        b = self.bottleneck(p4)

        # Upsample and Concatenate
        u1 = self.up1(b)
        u1 = self.pad_to_match(u1, e4)
        c1 = torch.cat([e4, u1], dim=1)
        d1 = self.dec1(c1)

        u2 = self.up2(d1)
        u2 = self.pad_to_match(u2, e3)
        c2 = torch.cat([e3, u2], dim=1)
        d2 = self.dec2(c2)

        u3 = self.up3(d2)
        u3 = self.pad_to_match(u3, e2)
        c3 = torch.cat([e2, u3], dim=1)
        d3 = self.dec3(c3)

        u4 = self.up4(d3)
        u4 = self.pad_to_match(u4, e1)
        c4 = torch.cat([e1, u4], dim=1)
        d4 = self.dec4(c4)

        # Output scores
        out = self.out_conv(d4)
        return out



## 3. Loss Functions and Training


In [4]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, num_classes=num_classes).permute(0, 2, 1).float()
        intersection = (probs * targets_onehot).sum(dim=(0, 2))
        union = probs.sum(dim=(0, 2)) + targets_onehot.sum(dim=(0, 2))
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

# Calculate class weights for heavily imbalanced data
all_labels = y_train.flatten()
class_counts = torch.bincount(all_labels)
weights = 1.0 / class_counts.float()
weights = weights / weights.sum() * len(weights)
print(f"Class Weights: {weights}")



Class Weights: tensor([0.1499, 1.8501])


In [5]:
# Create DataLoaders
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# For validation, we use a random 10% of the test set just to guide early stopping
val_size = int(0.1 * len(X_test))
test_size = len(X_test) - val_size
val_dataset, _ = torch.utils.data.random_split(TensorDataset(X_test, y_test), [val_size, test_size])
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = ECGUNet(in_channels=1, num_classes=2).to(device)
ce_loss = nn.CrossEntropyLoss(weight=weights.to(device))
dice_loss = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

num_epochs = 15
patience = 5
best_val_loss = float('inf')
counter = 0

print("Starting training...")
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss_ce = ce_loss(outputs, targets)
        loss_d = dice_loss(outputs, targets)
        loss = 0.5 * loss_ce + 0.5 * loss_d
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss_ce = ce_loss(outputs, targets)
            loss_d = dice_loss(outputs, targets)
            loss = 0.5 * loss_ce + 0.5 * loss_d
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "best_unet_mitbih.pth")
    else:
        counter += 1
        
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if counter >= patience:
        print("Early stopping triggered.")
        break
        
print(f"Training finished in {(time.time() - start_time)/60:.2f} minutes!")
model.load_state_dict(torch.load("best_unet_mitbih.pth"))



Starting training...
Epoch [1/15] Train Loss: 0.2129 | Val Loss: 0.0799
Epoch [2/15] Train Loss: 0.0415 | Val Loss: 0.0366
Epoch [3/15] Train Loss: 0.0233 | Val Loss: 0.0319
Epoch [4/15] Train Loss: 0.0183 | Val Loss: 0.0282
Epoch [5/15] Train Loss: 0.0167 | Val Loss: 0.0248
Epoch [6/15] Train Loss: 0.0139 | Val Loss: 0.0224
Epoch [7/15] Train Loss: 0.0132 | Val Loss: 0.0277
Epoch [8/15] Train Loss: 0.0124 | Val Loss: 0.0285
Epoch [9/15] Train Loss: 0.0125 | Val Loss: 0.0248
Epoch [10/15] Train Loss: 0.0123 | Val Loss: 0.0250
Epoch [11/15] Train Loss: 0.0095 | Val Loss: 0.0259
Early stopping triggered.
Training finished in 2.22 minutes!


C:\Users\AGNIVA SENGUPTA\AppData\Local\Temp\ipykernel_13820\111322344.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_unet_mitbih

<All keys matched successfully>

## 4. Evaluate on Test Set (DS2)
We will now load each record from the test set, chunk it, run inference, extract exact R-peak timestamps from the predicted segmentation masks, and evaluate accuracy.



In [6]:
def extract_peaks_from_mask(mask, offset=0):
    """
    Converts binary predicted mask blocks (e.g. [0, 1, 1, 1, 0]) into single peak timestamps
    by finding the center of each block.
    """
    peaks = []
    in_peak = False
    start_idx = 0
    for i in range(len(mask)):
        if mask[i] == 1 and not in_peak:
            in_peak = True
            start_idx = i
        elif mask[i] == 0 and in_peak:
            in_peak = False
            center = (start_idx + i) // 2
            peaks.append(center + offset)
    if in_peak:
        center = (start_idx + len(mask)) // 2
        peaks.append(center + offset)
    return peaks

def evaluate_peaks(detected_peaks, annotated_peaks, tolerance_samples=54): # 150ms at 360Hz = 54 samples
    tp = 0; fn = 0
    detected_matched = set()
    for ann in annotated_peaks:
        close_peaks = [p for p in detected_peaks if abs(p - ann) <= tolerance_samples]
        if close_peaks:
            closest = min(close_peaks, key=lambda x: abs(x - ann))
            if closest not in detected_matched:
                tp += 1
                detected_matched.add(closest)
            else:
                fn += 1
        else:
            fn += 1
    fp = len(detected_peaks) - len(detected_matched)
    return tp, fp, fn

results = []
model.eval()

print("Evaluating on DS2 Test Records...")
start_eval_time = time.time()

with torch.no_grad():
    for rec in DS2_TEST:
        if not os.path.exists(rec + '.dat'):
            continue
            
        record = wfdb.rdsamp(rec)
        ecg = record[0][:, 0]
        fs = record[1]['fs']
        ann_peaks = get_annotated_peaks(rec)
        
        det_peaks = []
        num_chunks = len(ecg) // CHUNK_SIZE
        
        t0 = time.time()
        for i in range(num_chunks):
            start = i * CHUNK_SIZE
            chunk_ecg = ecg[start:start+CHUNK_SIZE]
            
            std = np.std(chunk_ecg)
            if std > 0:
                chunk_ecg = (chunk_ecg - np.mean(chunk_ecg)) / std
            else:
                chunk_ecg = chunk_ecg - np.mean(chunk_ecg)
                
            chunk_tensor = torch.tensor(chunk_ecg, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            output = model(chunk_tensor)
            pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
            
            chunk_peaks = extract_peaks_from_mask(pred_mask, offset=start)
            det_peaks.extend(chunk_peaks)
            
        proc_time = time.time() - t0
        
        # We only evaluate up to the number of chunks we processed to be fair
        max_idx = num_chunks * CHUNK_SIZE
        valid_ann_peaks = [p for p in ann_peaks if p < max_idx]
        
        tp, fp, fn = evaluate_peaks(det_peaks, valid_ann_peaks)
        results.append({'Record': rec, 'Total Beats': len(valid_ann_peaks), 'TP': tp, 'FP': fp, 'FN': fn, 'Time (s)': proc_time})

print(f"Evaluation complete in {time.time()-start_eval_time:.2f} seconds.")



Evaluating on DS2 Test Records...
Evaluation complete in 84.13 seconds.


## 5. Summary Metrics


In [7]:
df = pd.DataFrame(results)

df['Sensitivity (%)'] = (df['TP'] / (df['TP'] + df['FN']) * 100).fillna(0)
df['Predictivity (%)'] = (df['TP'] / (df['TP'] + df['FP']) * 100).fillna(0)
df['Error Rate (%)'] = ((df['FP'] + df['FN']) / df['Total Beats'] * 100).fillna(0)

total_beats = df['Total Beats'].sum()
total_tp = df['TP'].sum()
total_fp = df['FP'].sum()
total_fn = df['FN'].sum()

overall_se = total_tp / (total_tp + total_fn) * 100
overall_pp = total_tp / (total_tp + total_fp) * 100
overall_er = (total_fp + total_fn) / total_beats * 100

print("="*50)
print("OVERALL PERFORMANCE (TEST SET / DS2)")
print("="*50)
print(f"Total Beats Evaluated : {total_beats}")
print(f"True Positives (TP)   : {total_tp}")
print(f"False Positives (FP)  : {total_fp}")
print(f"False Negatives (FN)  : {total_fn}")
print(f"Sensitivity (Se)      : {overall_se:.2f} %")
print(f"Positive Predictivity : {overall_pp:.2f} %")
print(f"Detection Error Rate  : {overall_er:.2f} %")
print("="*50)

df.to_csv('unet_mitbih_results.csv', index=False)
print("\nResults saved to 'unet_mitbih_results.csv'")
display(df.round(2))



OVERALL PERFORMANCE (TEST SET / DS2)
Total Beats Evaluated : 49651
True Positives (TP)   : 49609
False Positives (FP)  : 2101
False Negatives (FN)  : 42
Sensitivity (Se)      : 99.92 %
Positive Predictivity : 95.94 %
Detection Error Rate  : 4.32 %

Results saved to 'unet_mitbih_results.csv'


,Record,Total Beats,TP,FP,FN,Time (s),Sensitivity (%),Predictivity (%),Error Rate (%)
0,100,2269,2269,19,0,2.16,100.00,99.17,0.84
1,103,2082,2082,13,0,1.73,100.00,99.38,0.62
2,105,2569,2551,94,18,1.65,99.30,96.45,4.36
3,111,2122,2120,341,2,1.68,99.91,86.14,16.16
4,113,1792,1792,23,0,1.64,100.00,98.73,1.28
5,117,1533,1533,501,0,1.62,100.00,75.37,32.68
6,121,1860,1857,28,3,2.00,99.84,98.51,1.67
7,123,1516,1516,28,0,1.81,100.00,98.19,1.85
8,200,2598,2596,37,2,4.19,99.92,98.59,1.50
9,202,2133,2130,20,3,4.10,99.86,99.07,1.08
